# Imports

In [455]:
from math import log as log
from math import inf as inf
import numpy as np

# Constantes Globais

In [456]:
p = 0.5
N = 500 #definido arbitrariamente
MAXLEVEL = int(log(N, 1/p))

# TASKs

In [457]:
class Task:
    def __init__(self, atributos: dict):
        #att = ["task_id", "start_time", "end_times"]
        self.atributos = atributos

    def __getitem__(self, chave):
        return self.atributos[chave]

    def __setitem__(self, chave, valor):
        self.atributos[chave] = valor

# Skip List - ATIVIDADE B)

In [458]:
class No_skiplist:
    def __init__(self, nivel:int=0, tasks:Task = None):
        self.nivel = nivel
        self.frente = [None] * nivel
        self.conteudo = tasks

In [459]:
class NULL:
  def __init__(self):
    self.nivel = MAXLEVEL
    self.conteudo = {"task_id":inf, "start_time": inf, "end_time": inf}

In [460]:
class Skip_list:
  def __init__(self):
    self.tam = 0
    self.nivel_atual = 0
    self.NULL = NULL()
    self.cabeca = No_skiplist(nivel = MAXLEVEL,
                              tasks={"task_id":-inf, "start_time": -inf, "end_time": -inf})
    for i in range(len(self.cabeca.frente)):
      self.cabeca.frente[i] = self.NULL

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================

  def insert(self, chaveBusca: int, novoStartTime, novoEndTime) -> None:
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x != self.NULL and x.conteudo['task_id'] == chaveBusca:
      x.conteudo['start_time'] = novoStartTime
      x.conteudo['end_time'] = novoEndTime

    else:
      nvl = 0
      while np.random.random() < p and nvl < MAXLEVEL-1:
        nvl+=1

      if nvl > self.nivel_atual:
        for i in range(self.nivel_atual+1, nvl+1):
          atualiza[i] = self.cabeca
        self.nivel_atual = nvl

      task = Task({"task_id": chaveBusca, "start_time":
                   novoStartTime, "end_time": novoEndTime})

      x = No_skiplist(nvl+1, task)
      for i in range(x.nivel):
        x.frente[i] = self.NULL

      for i in range(nvl+1):
        x.frente[i] = atualiza[i].frente[i]
        atualiza[i].frente[i] = x

      self.tam+=1

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, chaveBusca):
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x!=self.NULL and x.conteudo['task_id'] == chaveBusca:
      for i in range(self.nivel_atual+1):
        if atualiza[i].frente[i] != x:
          break
        atualiza[i].frente[i] = x.frente[i]

      self.tam-=1

    while self.nivel_atual > 0 and self.cabeca.frente[self.nivel_atual] == self.NULL:
      self.nivel_atual-=1

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, chaveBusca):
    x = self.cabeca
    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]

    x = x.frente[0]
    if x.conteudo['task_id'] == chaveBusca:
      return True
    else:
      return False
#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self,chaveBuscaA, chaveBuscaB):
    x = self.cabeca
    y = self.cabeca
    output = [None]*(self.tam+1)
    nvly = self.nivel_atual

    for i in range(self.nivel_atual, -1, -1):
      while y.frente[i] != self.NULL and y.frente[i].conteudo['task_id'] <= chaveBuscaB:
        y = y.frente[i]
      nvly = i

    for j in range(nvly,-1,-1):
      while x.frente[j] != self.NULL and x.frente[j].conteudo['task_id'] < chaveBuscaA:
        x = x.frente[j]

    x = x.frente[0]
    c = 0

    if x.conteudo['task_id'] >= chaveBuscaA and y.conteudo['task_id'] <= chaveBuscaB:
      while x.conteudo['task_id'] <= y.conteudo['task_id']:
        output[c] = x.conteudo
        x = x.frente[0]
        c+=1

    return output

## EXEMPLO DE USO - SKIP LIST

In [461]:
skip = Skip_list()

### TESTE INSERT

In [462]:
#c = ["task_id", "start_time", "end_times"]
for i in range(MAXLEVEL):
  x = np.random.randint(0,100)
  y = np.random.randint(0,1000)
  z = np.random.randint(0,2000)
  # t = t = dict(zip(c, [x, y, z]))
  t = Task({"task_id":x,"start_time":y, 'end_time': z})
  skip.insert(x,y,z)


### VISUALIZAÇÃO SKIPLIST

In [463]:
def imprime_linear(skip:Skip_list):
  x = skip.cabeca.frente[0]
  while x != skip.NULL:
    print(x.conteudo.atributos)
    x = x.frente[0]

In [464]:
def imprime_estrutura(skip:Skip_list):
  x = skip.cabeca.frente[0]
  for i in range(skip.nivel_atual, -1, -1):
    print(f"--- Nível {i} ---")
    x = skip.cabeca.frente[i]

    while x != skip.NULL:
        print(x.conteudo.atributos['task_id'], end=" -> ")
        x = x.frente[i]
    print("NULL")

In [465]:
imprime_linear(skip)

{'task_id': 11, 'start_time': 328, 'end_time': 1090}
{'task_id': 20, 'start_time': 140, 'end_time': 1223}
{'task_id': 38, 'start_time': 324, 'end_time': 468}
{'task_id': 42, 'start_time': 570, 'end_time': 1384}
{'task_id': 49, 'start_time': 959, 'end_time': 1793}
{'task_id': 74, 'start_time': 45, 'end_time': 1503}
{'task_id': 75, 'start_time': 183, 'end_time': 590}


In [466]:
imprime_estrutura(skip)

--- Nível 4 ---
42 -> NULL
--- Nível 3 ---
42 -> 74 -> NULL
--- Nível 2 ---
11 -> 42 -> 74 -> NULL
--- Nível 1 ---
11 -> 42 -> 74 -> NULL
--- Nível 0 ---
11 -> 20 -> 38 -> 42 -> 49 -> 74 -> 75 -> NULL


## TESTE DELETE

In [467]:
skip.delete(5)

### VISUALIZAÇÃO SKIPLIST

In [468]:
imprime_linear(skip)

{'task_id': 11, 'start_time': 328, 'end_time': 1090}
{'task_id': 20, 'start_time': 140, 'end_time': 1223}
{'task_id': 38, 'start_time': 324, 'end_time': 468}
{'task_id': 42, 'start_time': 570, 'end_time': 1384}
{'task_id': 49, 'start_time': 959, 'end_time': 1793}
{'task_id': 74, 'start_time': 45, 'end_time': 1503}
{'task_id': 75, 'start_time': 183, 'end_time': 590}


In [469]:
imprime_estrutura(skip)

--- Nível 4 ---
42 -> NULL
--- Nível 3 ---
42 -> 74 -> NULL
--- Nível 2 ---
11 -> 42 -> 74 -> NULL
--- Nível 1 ---
11 -> 42 -> 74 -> NULL
--- Nível 0 ---
11 -> 20 -> 38 -> 42 -> 49 -> 74 -> 75 -> NULL


## Teste SEARCH

In [470]:
skip.search(26)

False

### VISUALIZAÇÃO SKIPLIST

In [471]:
imprime_linear(skip)

{'task_id': 11, 'start_time': 328, 'end_time': 1090}
{'task_id': 20, 'start_time': 140, 'end_time': 1223}
{'task_id': 38, 'start_time': 324, 'end_time': 468}
{'task_id': 42, 'start_time': 570, 'end_time': 1384}
{'task_id': 49, 'start_time': 959, 'end_time': 1793}
{'task_id': 74, 'start_time': 45, 'end_time': 1503}
{'task_id': 75, 'start_time': 183, 'end_time': 590}


In [472]:
imprime_estrutura(skip)

--- Nível 4 ---
42 -> NULL
--- Nível 3 ---
42 -> 74 -> NULL
--- Nível 2 ---
11 -> 42 -> 74 -> NULL
--- Nível 1 ---
11 -> 42 -> 74 -> NULL
--- Nível 0 ---
11 -> 20 -> 38 -> 42 -> 49 -> 74 -> 75 -> NULL


## Teste SEARCHINTERVAL

In [473]:
r = skip.searchinterval(0,0)

In [474]:
for i in range(len(r)):
  if r[i] != None:
    print(r[i].atributos)

### VISUALIZAÇÃO SKIPLIST

In [475]:
imprime_linear(skip)

{'task_id': 11, 'start_time': 328, 'end_time': 1090}
{'task_id': 20, 'start_time': 140, 'end_time': 1223}
{'task_id': 38, 'start_time': 324, 'end_time': 468}
{'task_id': 42, 'start_time': 570, 'end_time': 1384}
{'task_id': 49, 'start_time': 959, 'end_time': 1793}
{'task_id': 74, 'start_time': 45, 'end_time': 1503}
{'task_id': 75, 'start_time': 183, 'end_time': 590}


In [476]:
imprime_estrutura(skip)

--- Nível 4 ---
42 -> NULL
--- Nível 3 ---
42 -> 74 -> NULL
--- Nível 2 ---
11 -> 42 -> 74 -> NULL
--- Nível 1 ---
11 -> 42 -> 74 -> NULL
--- Nível 0 ---
11 -> 20 -> 38 -> 42 -> 49 -> 74 -> 75 -> NULL


# AVL - ATIVIDADE C)

In [477]:
class No_AVL:
  def __init__(self, esq=None, dir=None, task:Task=None):
    self.esq = esq
    self.dir = dir
    self.fator_balanceamento = 0
    self.altura = 0
    self.conteudo = task

In [478]:
class AVL:
  def __init__(self):
      pass
#===============================================================================
# ***************************** MAX ********************************************
#===============================================================================
  def MAX(self, h_esq, h_dir):
    if h_dir >= h_esq:
      return h_dir

    else:
      return h_esq

#===============================================================================
# ***************************** GET_ALTURA *************************************
#===============================================================================
  def getAltura(self, raiz:No_AVL):
    if raiz == None:
      return -1
    else:
      return raiz.altura

#===============================================================================
# ***************************** CALCULA_ALTURA *********************************
#===============================================================================
  def calculaAltura(self, raiz:No_AVL):
    h_esq = self.getAltura(raiz.esq)
    h_dir = self.getAltura(raiz.dir)

    return 1 + self.MAX(h_esq, h_dir)

#===============================================================================
# ************************** CALCULA_BALANCEAMENTO *****************************
#===============================================================================
  def calculaFatorBalanceamento(self, raiz:No_AVL):
    if raiz == None:
      return

    h_esq = self.getAltura(raiz.esq)
    h_dir = self.getAltura(raiz.dir)
    raiz.altura = self.calculaAltura(raiz)

    raiz.fator_balanceamento = h_esq - h_dir

#===============================================================================
# ***************************** ROTACAO_ESQUERDA *******************************
#===============================================================================
  def rotacaoESQUERDA(self, raiz:No_AVL):
    n_aux = raiz.dir

    if n_aux == None:
      return raiz

    t_aux = n_aux.esq

    n_aux.esq = raiz
    raiz.dir = t_aux

    self.calculaFatorBalanceamento(raiz)
    self.calculaFatorBalanceamento(n_aux)

    return n_aux

#===============================================================================
# ***************************** ROTACAO_DIREITA ********************************
#===============================================================================
  def rotacaoDIREITA(self,raiz:No_AVL):
    n_aux = raiz.esq

    if n_aux == None:
      return raiz

    t_aux = n_aux.dir

    n_aux.dir = raiz
    raiz.esq = t_aux

    self.calculaFatorBalanceamento(raiz)
    self.calculaFatorBalanceamento(n_aux)

    return n_aux

#===============================================================================
# ***************************** BALANCEAR **************************************
#===============================================================================
  def balancear(self, raiz:No_AVL):
    self.calculaFatorBalanceamento(raiz)

    #LL
    if raiz.fator_balanceamento > 1 and raiz.esq and raiz.esq.fator_balanceamento >= 0:
      return self.rotacaoDIREITA(raiz)

    #RR
    if raiz.fator_balanceamento < -1 and raiz.dir and raiz.dir.fator_balanceamento <= 0:
      return self.rotacaoESQUERDA(raiz)

    #LR
    if raiz.fator_balanceamento > 1 and raiz.esq and raiz.esq.fator_balanceamento < 0:
      raiz.esq = self.rotacaoESQUERDA(raiz.esq)
      return self.rotacaoDIREITA(raiz)

    #RL
    if raiz.fator_balanceamento < -1 and raiz.dir and raiz.dir.fator_balanceamento > 0:
      raiz.dir = self.rotacaoDIREITA(raiz.dir)
      return self.rotacaoESQUERDA(raiz)

    return raiz

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================
  def insert(self, raiz:No_AVL, task:Task):
    if raiz == None:
      novo_no = No_AVL(None, None, task)
      self.calculaFatorBalanceamento(novo_no)
      return novo_no

    if task['task_id'] < raiz.conteudo['task_id']:
      raiz.esq = self.insert(raiz.esq, task)

    else:
      raiz.dir = self.insert(raiz.dir, task)

    return self.balancear(raiz)

#===============================================================================
# ***************************** ANTECESSOR *************************************
#===============================================================================
  def antecessor(self, raiz:No_AVL):
    atual = raiz
    while atual.dir != None:
      atual = atual.dir
    return atual

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, raiz:No_AVL, chaveBusca):
    if raiz == None:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      raiz.esq = self.delete(raiz.esq, chaveBusca)

    elif chaveBusca > raiz.conteudo['task_id']:
      raiz.dir = self.delete(raiz.dir, chaveBusca)

    else:
      if raiz.esq == None:
        aux = raiz.dir
        raiz = None
        return aux

      elif raiz.dir == None:
        aux = raiz.esq
        raiz = None
        return aux

      aux = self.antecessor(raiz.esq)
      raiz.conteudo = aux.conteudo
      raiz.esq = self.delete(raiz.esq, aux.conteudo['task_id'])

    return self.balancear(raiz)

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, raiz:No_AVL, chaveBusca):
    if raiz == None or raiz.conteudo['task_id'] == chaveBusca:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      return self.search(raiz.esq, chaveBusca)

    elif chaveBusca > raiz.conteudo['task_id']:
      return self.search(raiz.dir, chaveBusca)

#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self, raiz:No_AVL, chaveBuscaA, chaveBuscaB, output:list):
    if raiz == None:
      return output

    if chaveBuscaA < raiz.conteudo['task_id']:
      self.searchinterval(raiz.esq, chaveBuscaA, chaveBuscaB, output)

    if chaveBuscaA <= raiz.conteudo['task_id'] <= chaveBuscaB:
      output.append(raiz)

    if chaveBuscaB > raiz.conteudo['task_id']:
      self.searchinterval(raiz.dir, chaveBuscaA, chaveBuscaB, output)

    return output



## EXEMPLO DE USO - AVL

In [479]:
AVL = AVL()

In [480]:
raiz = None
for i in range(MAXLEVEL):
  x = np.random.randint(0,100)
  y = np.random.randint(0,1000)
  z = np.random.randint(1000,2000)
  t = Task({"task_id":x,"start_time":y, 'end_time': z})
  raiz=AVL.insert(raiz,t)

### Teste INSERT

In [481]:
def percursoIN_ORDER(raiz):
  if raiz == None:
    return
  percursoIN_ORDER(raiz.esq)
  print(raiz.conteudo.atributos)
  percursoIN_ORDER(raiz.dir)

### Visualização

In [482]:
percursoIN_ORDER(raiz)

{'task_id': 1, 'start_time': 11, 'end_time': 1608}
{'task_id': 4, 'start_time': 512, 'end_time': 1142}
{'task_id': 15, 'start_time': 385, 'end_time': 1477}
{'task_id': 16, 'start_time': 965, 'end_time': 1051}
{'task_id': 30, 'start_time': 57, 'end_time': 1268}
{'task_id': 43, 'start_time': 868, 'end_time': 1448}
{'task_id': 87, 'start_time': 181, 'end_time': 1456}
{'task_id': 97, 'start_time': 875, 'end_time': 1535}


### Teste DELETE

In [483]:
raiz=AVL.delete(raiz, 79)

### Visualização

In [484]:
percursoIN_ORDER(raiz)

{'task_id': 1, 'start_time': 11, 'end_time': 1608}
{'task_id': 4, 'start_time': 512, 'end_time': 1142}
{'task_id': 15, 'start_time': 385, 'end_time': 1477}
{'task_id': 16, 'start_time': 965, 'end_time': 1051}
{'task_id': 30, 'start_time': 57, 'end_time': 1268}
{'task_id': 43, 'start_time': 868, 'end_time': 1448}
{'task_id': 87, 'start_time': 181, 'end_time': 1456}
{'task_id': 97, 'start_time': 875, 'end_time': 1535}


### Teste SEARCH

In [485]:
busca=AVL.search(raiz,67)

In [486]:
if busca:
  print(busca.conteudo.atributos)
else:
  print("FALSO")

FALSO


## Teste SEARCHINTERVAL

In [487]:
output = []

In [488]:
buscaIntervalo = AVL.searchinterval(raiz,0,60, output)
for i in buscaIntervalo:
  print(i.conteudo.atributos)

{'task_id': 1, 'start_time': 11, 'end_time': 1608}
{'task_id': 4, 'start_time': 512, 'end_time': 1142}
{'task_id': 15, 'start_time': 385, 'end_time': 1477}
{'task_id': 16, 'start_time': 965, 'end_time': 1051}
{'task_id': 30, 'start_time': 57, 'end_time': 1268}
{'task_id': 43, 'start_time': 868, 'end_time': 1448}


# ATIVIDADE - D)
Adapted from: https://github.com/amitjoshi2724/gpi-job-scheduling
## corpus sintético

In [489]:
import random

## Geração de Tasks - Especificação da Atividade

In [491]:
def generate_uniform_tasks(n):

    MAX_VAL = 10**6
    tasks = [(random.randint(0, MAX_VAL), random.randint(0, MAX_VAL), id) for id in range(n)]
    tasks = [{'task_id': id,
              'start_time': min(s, e),
              'end_time': max(s, e)} for s, e, id in tasks]

    return tasks

t_uniform = generate_uniform_tasks(100)

In [492]:
def generate_normal_tasks(n):

    K = 10**9
    sigma = K / 10

    # Truncated normal start times in [0, K]
    start_times = np.clip(np.random.normal(loc=K/2, scale=sigma, size=n), 0, K)
    durations = np.random.uniform(1.0, 10**6, size=n)
    end_times = start_times + durations
    tasks = [{'task_id': id,
              'start_time': s,
              'end_time': e} for id, (s, e) in enumerate(zip(start_times, end_times))]

    return tasks

t_normal = generate_normal_tasks(100)

In [493]:
def generate_zipf_tasks(n):

    K = 10**9  # Upper bound of time domain

    # Clustered start times near 0 using truncated exponential
    scale = K / 10  # Controls clustering tightness; lower = tighter near 0
    start_times = np.random.exponential(scale=scale, size=n)
    start_times = np.clip(start_times, 0, K)  # Ensure within [0, K]

    # Zipf durations, scaled and capped
    raw_zipf = np.random.zipf(a=2.0, size=n)
    durations = np.minimum(100 * raw_zipf, 10**6)
    end_times = start_times + durations

    tasks = [{'task_id': id,
              'start_time': float(s),
              'end_time': float(e)} for id, (s, e) in enumerate(zip(start_times, end_times))]

    return tasks

t_zipf = generate_zipf_tasks(100)

## Adaptação SearchInterval SKIP LIST
A implementação anterior era mais genérica se baseando apenas no valor da chave ($task_{id}$)

Nesta implementação, adaptei para atender ao *timestamp* ($start\_time \geq chaveBuscaA$ e $chaveBuscaB \geq end\_time$)

Contudo, precisei fazer algumas correções por que a forma anterior funcionava para quando queríamos comparar $chaveBuscaA$ ou $chaveBuscaB$ apenas com $task\_id$

Nesta nova versão, descarto tudo o que usava $y$ (apontador superior para definir fim do intervalo) e uso apenas $x$ para percorrer a estrutura uma única vez e mais outra vez para gerar a saída

In [494]:
class No_skiplist:
    def __init__(self, nivel:int=0, tasks:Task = None):
        self.nivel = nivel
        self.frente = [None] * nivel
        self.conteudo = tasks

In [495]:
class NULL:
  def __init__(self):
    self.nivel = MAXLEVEL
    self.conteudo = {"task_id":inf, "start_time": inf, "end_time": inf}

In [496]:
class Skip_list:
  def __init__(self):
    self.tam = 0
    self.nivel_atual = 0
    self.NULL = NULL()
    self.cabeca = No_skiplist(nivel = MAXLEVEL,
                              tasks={"task_id":-inf, "start_time": -inf, "end_time": -inf})
    for i in range(len(self.cabeca.frente)):
      self.cabeca.frente[i] = self.NULL

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================

  def insert(self, chaveBusca: int, novoStartTime, novoEndTime) -> None:
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x != self.NULL and x.conteudo['task_id'] == chaveBusca:
      x.conteudo['start_time'] = novoStartTime
      x.conteudo['end_time'] = novoEndTime

    else:
      nvl = 0
      while np.random.random() < p and nvl < MAXLEVEL-1:
        nvl+=1

      if nvl > self.nivel_atual:
        for i in range(self.nivel_atual+1, nvl+1):
          atualiza[i] = self.cabeca
        self.nivel_atual = nvl

      task = Task({"task_id": chaveBusca, "start_time":
                   novoStartTime, "end_time": novoEndTime})

      x = No_skiplist(nvl+1, task)
      for i in range(x.nivel):
        x.frente[i] = self.NULL

      for i in range(nvl+1):
        x.frente[i] = atualiza[i].frente[i]
        atualiza[i].frente[i] = x

      self.tam+=1

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, chaveBusca):
    atualiza = [None]*MAXLEVEL
    x = self.cabeca

    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]
      atualiza[i] = x

    x = x.frente[0]
    if x!=self.NULL and x.conteudo['task_id'] == chaveBusca:
      for i in range(self.nivel_atual+1):
        if atualiza[i].frente[i] != x:
          break
        atualiza[i].frente[i] = x.frente[i]

      self.tam-=1

    while self.nivel_atual > 0 and self.cabeca.frente[self.nivel_atual] == self.NULL:
      self.nivel_atual-=1

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, chaveBusca):
    x = self.cabeca
    for i in range(self.nivel_atual, -1, -1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['task_id'] < chaveBusca:
        x = x.frente[i]

    x = x.frente[0]
    if x.conteudo['task_id'] == chaveBusca:
      return True
    else:
      return False
#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self,chaveBuscaA, chaveBuscaB):
    x = self.cabeca
    output = [None]*(self.tam+1)

    for i in range(self.nivel_atual, -1,-1):
      while x.frente[i] != self.NULL and x.frente[i].conteudo['start_time'] < chaveBuscaA:
        x=x.frente[i]

    x = x.frente[0]
    c = 0
    while x != self.NULL:
      if chaveBuscaA <= x.conteudo['start_time'] and x.conteudo['end_time'] <= chaveBuscaB:
        output[c] = x.conteudo
        c+=1
      x = x.frente[0]

    return output

### Teste Novo SearchInterval

In [497]:
s = Skip_list()

In [498]:
for i in range(MAXLEVEL):
  x = np.random.randint(0,100)
  y = np.random.randint(0,1000)
  z = np.random.randint(1000,2000)
  # t = t = dict(zip(c, [x, y, z]))
  t = Task({"task_id":x,"start_time":y, 'end_time': z})
  s.insert(x,y,z)

In [499]:
v = s.searchinterval(200,1500)

In [500]:
for i in range(len(v)):
  if v[i] != None:
    print(v[i].atributos)

{'task_id': 89, 'start_time': 774, 'end_time': 1272}


#### VISUALIZAÇÃO SKIPLIST

In [501]:
imprime_linear(s)

{'task_id': 11, 'start_time': 186, 'end_time': 1253}
{'task_id': 15, 'start_time': 492, 'end_time': 1597}
{'task_id': 16, 'start_time': 438, 'end_time': 1518}
{'task_id': 75, 'start_time': 26, 'end_time': 1946}
{'task_id': 89, 'start_time': 774, 'end_time': 1272}
{'task_id': 90, 'start_time': 572, 'end_time': 1694}


## Adaptações AVL - SearchInterval
A adaptação, assim como na *skip list*, foi o uso de $start\_time$ e $end\_time$ ao invés de $task\_id$

De resto a estrutura do algoritmo se manteve a mesma, apenas mudanças nas comparações do *if*

In [502]:
class AVL:
  def __init__(self):
      pass
#===============================================================================
# ***************************** MAX ********************************************
#===============================================================================
  def MAX(self, h_esq, h_dir):
    if h_dir >= h_esq:
      return h_dir

    else:
      return h_esq

#===============================================================================
# ***************************** GET_ALTURA *************************************
#===============================================================================
  def getAltura(self, raiz:No_AVL):
    if raiz == None:
      return -1
    else:
      return raiz.altura

#===============================================================================
# ***************************** CALCULA_ALTURA *********************************
#===============================================================================
  def calculaAltura(self, raiz:No_AVL):
    h_esq = self.getAltura(raiz.esq)
    h_dir = self.getAltura(raiz.dir)

    return 1 + self.MAX(h_esq, h_dir)

#===============================================================================
# ************************** CALCULA_BALANCEAMENTO *****************************
#===============================================================================
  def calculaFatorBalanceamento(self, raiz:No_AVL):
    if raiz == None:
      return

    h_esq = self.getAltura(raiz.esq)
    h_dir = self.getAltura(raiz.dir)
    raiz.altura = self.calculaAltura(raiz)

    raiz.fator_balanceamento = h_esq - h_dir

#===============================================================================
# ***************************** ROTACAO_ESQUERDA *******************************
#===============================================================================
  def rotacaoESQUERDA(self, raiz:No_AVL):
    n_aux = raiz.dir

    if n_aux == None:
      return raiz

    t_aux = n_aux.esq

    n_aux.esq = raiz
    raiz.dir = t_aux

    self.calculaFatorBalanceamento(raiz)
    self.calculaFatorBalanceamento(n_aux)

    return n_aux

#===============================================================================
# ***************************** ROTACAO_DIREITA ********************************
#===============================================================================
  def rotacaoDIREITA(self,raiz:No_AVL):
    n_aux = raiz.esq

    if n_aux == None:
      return raiz

    t_aux = n_aux.dir

    n_aux.dir = raiz
    raiz.esq = t_aux

    self.calculaFatorBalanceamento(raiz)
    self.calculaFatorBalanceamento(n_aux)

    return n_aux

#===============================================================================
# ***************************** BALANCEAR **************************************
#===============================================================================
  def balancear(self, raiz:No_AVL):
    self.calculaFatorBalanceamento(raiz)

    #LL
    if raiz.fator_balanceamento > 1 and raiz.esq and raiz.esq.fator_balanceamento >= 0:
      return self.rotacaoDIREITA(raiz)

    #RR
    if raiz.fator_balanceamento < -1 and raiz.dir and raiz.dir.fator_balanceamento <= 0:
      return self.rotacaoESQUERDA(raiz)

    #LR
    if raiz.fator_balanceamento > 1 and raiz.esq and raiz.esq.fator_balanceamento < 0:
      raiz.esq = self.rotacaoESQUERDA(raiz.esq)
      return self.rotacaoDIREITA(raiz)

    #RL
    if raiz.fator_balanceamento < -1 and raiz.dir and raiz.dir.fator_balanceamento > 0:
      raiz.dir = self.rotacaoDIREITA(raiz.dir)
      return self.rotacaoESQUERDA(raiz)

    return raiz

#===============================================================================
# ***************************** INSERT *****************************************
#===============================================================================
  def insert(self, raiz:No_AVL, task:Task):
    if raiz == None:
      novo_no = No_AVL(None, None, task)
      self.calculaFatorBalanceamento(novo_no)
      return novo_no

    if task['task_id'] < raiz.conteudo['task_id']:
      raiz.esq = self.insert(raiz.esq, task)

    else:
      raiz.dir = self.insert(raiz.dir, task)

    return self.balancear(raiz)

#===============================================================================
# ***************************** ANTECESSOR *************************************
#===============================================================================
  def antecessor(self, raiz:No_AVL):
    atual = raiz
    while atual.dir != None:
      atual = atual.dir
    return atual

#===============================================================================
# ***************************** DELETE *****************************************
#===============================================================================
  def delete(self, raiz:No_AVL, chaveBusca):
    if raiz == None:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      raiz.esq = self.delete(raiz.esq, chaveBusca)

    elif chaveBusca > raiz.conteudo['task_id']:
      raiz.dir = self.delete(raiz.dir, chaveBusca)

    else:
      if raiz.esq == None:
        aux = raiz.dir
        raiz = None
        return aux

      elif raiz.dir == None:
        aux = raiz.esq
        raiz = None
        return aux

      aux = self.antecessor(raiz.esq)
      raiz.conteudo = aux.conteudo
      raiz.esq = self.delete(raiz.esq, aux.conteudo['task_id'])

    return self.balancear(raiz)

#===============================================================================
# ***************************** SEARCH *****************************************
#===============================================================================
  def search(self, raiz:No_AVL, chaveBusca):
    if raiz == None or raiz.conteudo['task_id'] == chaveBusca:
      return raiz

    if chaveBusca < raiz.conteudo['task_id']:
      return self.search(raiz.esq, chaveBusca)

    elif chaveBusca > raiz.conteudo['task_id']:
      return self.search(raiz.dir, chaveBusca)

#===============================================================================
# ***************************** SEARCHINTERVAL *********************************
#===============================================================================
  def searchinterval(self, raiz:No_AVL, chaveBuscaA, chaveBuscaB, output:list):
    if raiz == None:
      return output

    if chaveBuscaA < raiz.conteudo['start_time']:
      self.searchinterval(raiz.dir, chaveBuscaA, chaveBuscaB, output)

    if chaveBuscaA <= raiz.conteudo['start_time'] and raiz.conteudo['end_time'] <= chaveBuscaB:
      output.append(raiz)

    if chaveBuscaB > raiz.conteudo['end_time']:
      self.searchinterval(raiz.esq, chaveBuscaA, chaveBuscaB, output)

    return output

### Teste Novo SearchInterval

In [503]:
avl = AVL()

In [504]:
root = None
for i in range(MAXLEVEL):
  x = np.random.randint(0,100)
  y = np.random.randint(0,1000)
  z = np.random.randint(1000,2000)
  t = Task({"task_id":x,"start_time":y, 'end_time': z})
  root=avl.insert(root,t)

In [505]:
output = []
buscaIntervalo = avl.searchinterval(raiz,600,1800, output)
for i in buscaIntervalo:
  print(i.conteudo.atributos)

{'task_id': 43, 'start_time': 868, 'end_time': 1448}
{'task_id': 16, 'start_time': 965, 'end_time': 1051}


### Visualização

In [506]:
percursoIN_ORDER(raiz)

{'task_id': 1, 'start_time': 11, 'end_time': 1608}
{'task_id': 4, 'start_time': 512, 'end_time': 1142}
{'task_id': 15, 'start_time': 385, 'end_time': 1477}
{'task_id': 16, 'start_time': 965, 'end_time': 1051}
{'task_id': 30, 'start_time': 57, 'end_time': 1268}
{'task_id': 43, 'start_time': 868, 'end_time': 1448}
{'task_id': 87, 'start_time': 181, 'end_time': 1456}
{'task_id': 97, 'start_time': 875, 'end_time': 1535}


## Distribuição Uniforme - Skip List

## Distribuição Normal - Skip List

## Distribuição ZIPF - Skip List

## Distribuição Uniforme - AVL

## Distribuição Normal - AVL

## Distribuição ZIPF - AVL